In [1]:

from google.colab import files
uploaded = files.upload()


Saving archive.zip to archive.zip


In [2]:
import zipfile
import pandas as pd

with zipfile.ZipFile("archive.zip", "r") as zip_ref:
    zip_ref.extractall(".")

df_books = pd.read_csv("books.csv", on_bad_lines="skip")

print(f"✅ Dataset carregado!")
print(f"Total de livros: {len(df_books)}")
print(f"Colunas: {list(df_books.columns)}")
df_books.head()

✅ Dataset carregado!
Total de livros: 11123
Colunas: ['bookID', 'title', 'authors', 'average_rating', 'isbn', 'isbn13', 'language_code', '  num_pages', 'ratings_count', 'text_reviews_count', 'publication_date', 'publisher']


,bookID,title,authors,average_rating,isbn,isbn13,language_code,num_pages,ratings_count,text_reviews_count,publication_date,publisher
0,1,Harry Potter and the Half-Blood Prince (Harry ...,J.K. Rowling/Mary GrandPré,4.57,0439785960,9780439785969,eng,652,2095690,27591,9/16/2006,Scholastic Inc.
1,2,Harry Potter and the Order of the Phoenix (Har...,J.K. Rowling/Mary GrandPré,4.49,0439358078,9780439358071,eng,870,2153167,29221,9/1/2004,Scholastic Inc.
2,4,Harry Potter and the Chamber of Secrets (Harry...,J.K. Rowling,4.42,0439554896,9780439554893,eng,352,6333,244,11/1/2003,Scholastic
3,5,Harry Potter and the Prisoner of Azkaban (Harr...,J.K. Rowling/Mary GrandPré,4.56,043965548X,9780439655484,eng,435,2339585,36325,5/1/2004,Scholastic Inc.
4,8,Harry Potter Boxed Set Books 1-5 (Harry Potte...,J.K. Rowling/Mary GrandPré,4.78,0439682584,9780439682589,eng,2690,41428,164,9/13/2004,Scholastic


In [4]:

df_books.columns = df_books.columns.str.strip()

print("✅ Colunas corrigidas:")
print(list(df_books.columns))

✅ Colunas corrigidas:
['bookID', 'title', 'authors', 'average_rating', 'isbn', 'isbn13', 'language_code', 'num_pages', 'ratings_count', 'text_reviews_count', 'publication_date', 'publisher']


In [5]:
print("📐 Tamanho do dataset:")
print(f"{df_books.shape[0]} livros e {df_books.shape[1]} colunas\n")

print("📋 Tipos de dados:")
print(df_books.dtypes)

print("\n🔍 Valores faltantes:")
print(df_books.isnull().sum())

print("\n📊 Estatísticas gerais:")
df_books[["average_rating", "num_pages", "ratings_count", "text_reviews_count"]].describe()

📐 Tamanho do dataset:
11123 livros e 12 colunas

📋 Tipos de dados:
bookID                  int64
title                  object
authors                object
average_rating        float64
isbn                   object
isbn13                  int64
language_code          object
num_pages               int64
ratings_count           int64
text_reviews_count      int64
publication_date       object
publisher              object
dtype: object

🔍 Valores faltantes:
bookID                0
title                 0
authors               0
average_rating        0
isbn                  0
isbn13                0
language_code         0
num_pages             0
ratings_count         0
text_reviews_count    0
publication_date      0
publisher             0
dtype: int64

📊 Estatísticas gerais:


,average_rating,num_pages,ratings_count,text_reviews_count
count,11123.000000,11123.000000,1.112300e+04,11123.000000
mean,3.934075,336.405556,1.794285e+04,542.048099
std,0.350485,241.152626,1.124992e+05,2576.619589
min,0.000000,0.000000,0.000000e+00,0.000000
25%,3.770000,192.000000,1.040000e+02,9.000000
50%,3.960000,299.000000,7.450000e+02,47.000000
75%,4.140000,416.000000,5.000500e+03,238.000000
max,5.000000,6576.000000,4.597666e+06,94265.000000


In [6]:

df_books = df_books[df_books["ratings_count"] > 0]
df_books = df_books[df_books["average_rating"] > 0]
df_books = df_books[df_books["num_pages"] > 0]


df_books["publication_date"] = pd.to_datetime(df_books["publication_date"], errors="coerce")
df_books["ano_publicacao"] = df_books["publication_date"].dt.year


df_books = df_books[df_books["language_code"].isin(["eng", "en-US", "en-GB"])]

import numpy as np
df_books["log_ratings"] = np.log1p(df_books["ratings_count"])

print(f"✅ Dataset limpo!")
print(f"Total de livros após limpeza: {len(df_books)}")
print(f"\nPeríodo: {int(df_books['ano_publicacao'].min())} até {int(df_books['ano_publicacao'].max())}")
print(f"\nTop 5 livros mais avaliados:")
df_books.nlargest(5, "ratings_count")[["title", "authors", "average_rating", "ratings_count"]]

✅ Dataset limpo!
Total de livros após limpeza: 10395

Período: 1900 até 2020

Top 5 livros mais avaliados:


,title,authors,average_rating,ratings_count
10336,Twilight (Twilight #1),Stephenie Meyer,3.59,4597666
1697,The Hobbit or There and Back Again,J.R.R. Tolkien,4.27,2530894
1462,The Catcher in the Rye,J.D. Salinger,3.80,2457092
307,Angels & Demons (Robert Langdon #1),Dan Brown,3.89,2418736
3,Harry Potter and the Prisoner of Azkaban (Harr...,J.K. Rowling/Mary GrandPré,4.56,2339585


In [7]:
import sqlite3


conn = sqlite3.connect("livros.db")

df_books.to_sql("livros", conn, if_exists="replace", index=False)

print("✅ Banco de dados criado!")

query = """
SELECT
    title,
    authors,
    average_rating,
    ratings_count,
    num_pages
FROM livros
ORDER BY ratings_count DESC
LIMIT 10
"""

resultado = pd.read_sql_query(query, conn)
print("\n📚 Top 10 livros mais avaliados via SQL:")
resultado

✅ Banco de dados criado!

📚 Top 10 livros mais avaliados via SQL:


,title,authors,average_rating,ratings_count,num_pages
0,Twilight (Twilight #1),Stephenie Meyer,3.59,4597666,501
1,The Hobbit or There and Back Again,J.R.R. Tolkien,4.27,2530894,366
2,The Catcher in the Rye,J.D. Salinger,3.80,2457092,277
3,Angels & Demons (Robert Langdon #1),Dan Brown,3.89,2418736,736
4,Harry Potter and the Prisoner of Azkaban (Harr...,J.K. Rowling/Mary GrandPré,4.56,2339585,435
5,Harry Potter and the Chamber of Secrets (Harry...,J.K. Rowling/Mary GrandPré,4.42,2293963,341
6,Harry Potter and the Order of the Phoenix (Har...,J.K. Rowling/Mary GrandPré,4.49,2153167,870
7,The Fellowship of the Ring (The Lord of the Ri...,J.R.R. Tolkien,4.36,2128944,398
8,Animal Farm,George Orwell/Boris Grabnar/Peter Škerl,3.93,2111750,122
9,Harry Potter and the Half-Blood Prince (Harry ...,J.K. Rowling/Mary GrandPré,4.57,2095690,652


In [8]:

query1 = """
SELECT
    ROUND(average_rating) as nota_arredondada,
    COUNT(*) as total_livros,
    ROUND(AVG(ratings_count)) as media_avaliacoes,
    ROUND(AVG(num_pages)) as media_paginas
FROM livros
GROUP BY nota_arredondada
ORDER BY nota_arredondada
"""

print("📊 QUERY 1 — Nota vs Popularidade:")
pd.read_sql_query(query1, conn)

📊 QUERY 1 — Nota vs Popularidade:


,nota_arredondada,total_livros,media_avaliacoes,media_paginas
0,1.0,2,2.0,71.0
1,2.0,8,679.0,366.0
2,3.0,657,4936.0,271.0
3,4.0,9530,19861.0,339.0
4,5.0,198,27463.0,506.0


In [9]:

query2 = """
SELECT
    publisher,
    COUNT(*) as total_livros,
    ROUND(AVG(average_rating), 2) as nota_media,
    ROUND(AVG(ratings_count)) as media_avaliacoes
FROM livros
GROUP BY publisher
HAVING total_livros >= 20
ORDER BY total_livros DESC
LIMIT 10
"""

print("🏢 QUERY 2 — Top editoras:")
pd.read_sql_query(query2, conn)

🏢 QUERY 2 — Top editoras:


,publisher,total_livros,nota_media,media_avaliacoes
0,Vintage,317,3.89,15768.0
1,Penguin Books,261,3.92,42021.0
2,Penguin Classics,183,3.95,28855.0
3,Mariner Books,146,3.93,9465.0
4,Ballantine Books,142,3.88,21229.0
5,HarperCollins,112,4.04,25406.0
6,Pocket Books,111,3.90,34984.0
7,Harper Perennial,110,3.90,19997.0
8,Bantam,110,3.89,35903.0
9,VIZ Media LLC,88,4.24,10117.0


In [10]:

query3 = """
SELECT
    (ano_publicacao / 10) * 10 as decada,
    COUNT(*) as total_livros,
    ROUND(AVG(average_rating), 2) as nota_media,
    ROUND(AVG(ratings_count)) as media_avaliacoes
FROM livros
WHERE ano_publicacao IS NOT NULL
GROUP BY decada
ORDER BY decada
"""

print("📅 QUERY 3 — Livros por década:")
pd.read_sql_query(query3, conn)

📅 QUERY 3 — Livros por década:


,decada,total_livros,nota_media,media_avaliacoes
0,1900.0,1,3.88,332.0
1,1913.0,1,3.96,111.0
2,1923.0,1,4.29,35.0
3,1925.0,2,3.96,84.0
4,1928.0,1,4.34,110.0
...,...,...,...,...
78,2016.0,3,3.99,6200.0
79,2017.0,6,3.99,940.0
80,2018.0,3,4.10,1507.0
81,2019.0,6,3.92,16121.0


In [11]:
import plotly.express as px
import plotly.graph_objects as go

query3_corrigida = """
SELECT
    (CAST(ano_publicacao AS INTEGER) / 10) * 10 as decada,
    COUNT(*) as total_livros,
    ROUND(AVG(average_rating), 2) as nota_media,
    ROUND(AVG(ratings_count)) as media_avaliacoes
FROM livros
WHERE ano_publicacao >= 1950 AND ano_publicacao <= 2010
GROUP BY decada
ORDER BY decada
"""

df_decadas = pd.read_sql_query(query3_corrigida, conn)


fig1 = px.bar(
    df_decadas,
    x="decada",
    y="total_livros",
    color="nota_media",
    title="📅 Livros publicados por década — quantidade e nota média",
    labels={
        "decada"       : "Década",
        "total_livros" : "Total de Livros",
        "nota_media"   : "Nota Média"
    },
    color_continuous_scale="RdYlGn",
    text="total_livros",
)
fig1.update_traces(textposition="outside")
fig1.update_layout(height=500, plot_bgcolor="white")
fig1.show()

fig2 = px.scatter(
    df_decadas,
    x="nota_media",
    y="media_avaliacoes",
    size="total_livros",
    text="decada",
    title="🫧 Nota média vs Popularidade por década",
    labels={
        "nota_media"        : "Nota Média",
        "media_avaliacoes"  : "Média de Avaliações",
        "total_livros"      : "Total de Livros"
    },
    color="decada",
    size_max=50,
)
fig2.update_traces(textposition="top center")
fig2.update_layout(height=500, plot_bgcolor="white")
fig2.show()

In [12]:

from plotly.subplots import make_subplots

fig1_novo = make_subplots(specs=[[{"secondary_y": True}]])


fig1_novo.add_trace(
    go.Bar(
        x=df_decadas["decada"],
        y=df_decadas["total_livros"],
        name="Total de Livros",
        marker_color="#4CAF50",
        opacity=0.7,
    ),
    secondary_y=False,
)


fig1_novo.add_trace(
    go.Scatter(
        x=df_decadas["decada"],
        y=df_decadas["nota_media"],
        name="Nota Média",
        mode="lines+markers",
        line=dict(color="#FF5722", width=3),
        marker=dict(size=8),
    ),
    secondary_y=True,
)

fig1_novo.update_layout(
    title="📅 Quantidade de livros e nota média por década",
    height=500,
    plot_bgcolor="white",
    xaxis=dict(showgrid=True, gridcolor="#f0f0f0", tickmode="linear", dtick=10),
)
fig1_novo.update_yaxes(title_text="Total de Livros", secondary_y=False)
fig1_novo.update_yaxes(title_text="Nota Média", secondary_y=True, range=[3.88, 4.05])
fig1_novo.show()


fig2_novo = px.bar(
    df_decadas.sort_values("media_avaliacoes", ascending=True),
    x="media_avaliacoes",
    y="decada",
    orientation="h",
    title="📊 Média de avaliações por década — livros de qual época são mais lidos hoje?",
    labels={
        "media_avaliacoes" : "Média de Avaliações",
        "decada"           : "Década"
    },
    color="media_avaliacoes",
    color_continuous_scale="Blues",
    text="media_avaliacoes",
)
fig2_novo.update_traces(texttemplate="%{text:,.0f}", textposition="outside")
fig2_novo.update_layout(height=450, plot_bgcolor="white")
fig2_novo.show()

In [15]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error
import numpy as np


features = ["average_rating", "num_pages", "ano_publicacao", "text_reviews_count"]
target = "log_ratings"

df_modelo = df_books[features + [target]].dropna()

X = df_modelo[features]
y = df_modelo[target]


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


modelo = LinearRegression()
modelo.fit(X_train, y_train)


y_pred = modelo.predict(X_test)
r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)

print("🤖 MODELO DE REGRESSÃO LINEAR")
print("=" * 40)
print(f"R² Score: {r2:.3f} ({r2*100:.1f}% da variação explicada)")
print(f"Erro médio absoluto: {mae:.3f}")
print("\n📊 Impacto de cada variável na popularidade:")
for feature, coef in zip(features, modelo.coef_):
    print(f"  {feature}: {coef:+.4f}")
print(f"\n  Intercepto: {modelo.intercept_:.4f}")

🤖 MODELO DE REGRESSÃO LINEAR
R² Score: 0.179 (17.9% da variação explicada)
Erro médio absoluto: 1.850

📊 Impacto de cada variável na popularidade:
  average_rating: +0.7709
  num_pages: +0.0011
  ano_publicacao: +0.0352
  text_reviews_count: +0.0004

  Intercepto: -67.1536


In [16]:

coeficientes = pd.DataFrame({
    "variavel" : features,
    "impacto"  : modelo.coef_
}).sort_values("impacto", ascending=True)

fig3 = px.bar(
    coeficientes,
    x="impacto",
    y="variavel",
    orientation="h",
    title="🤖 O que mais influencia a popularidade de um livro?",
    labels={"impacto": "Impacto na Popularidade", "variavel": "Variável"},
    color="impacto",
    color_continuous_scale="RdYlGn",
)
fig3.update_layout(height=400, plot_bgcolor="white")
fig3.add_vline(x=0, line_dash="dash", line_color="gray")
fig3.show()


fig4 = px.scatter(
    x=y_test,
    y=y_pred,
    title="📈 Valores reais vs previstos pelo modelo",
    labels={"x": "Popularidade Real", "y": "Popularidade Prevista"},
    opacity=0.4,
    color_discrete_sequence=["#2196F3"],
)
fig4.add_shape(
    type="line",
    x0=y_test.min(), y0=y_test.min(),
    x1=y_test.max(), y1=y_test.max(),
    line=dict(color="red", dash="dash"),
)
fig4.update_layout(height=450, plot_bgcolor="white")
fig4.show()

In [17]:

destaques = [
    "Twilight (Twilight #1)",
    "The Hobbit or There and Back Again",
    "The Catcher in the Rye",
    "Harry Potter and the Prisoner of Azkaban (Harry Potter  #3)",
    "Animal Farm",
    "Angels & Demons (Robert Langdon #1)",
]

df_books["destaque"] = df_books["title"].apply(
    lambda x: x[:20] + "..." if x in destaques else ""
)

fig5 = px.scatter(
    df_books[df_books["ratings_count"] > 100],
    x="average_rating",
    y="log_ratings",
    opacity=0.3,
    color_discrete_sequence=["#2196F3"],
    title="📚 Nota vs Popularidade — cada ponto é um livro",
    labels={
        "average_rating" : "Nota Média (Goodreads)",
        "log_ratings"    : "Popularidade (log de avaliações)",
    },
)


df_dest = df_books[df_books["title"].isin(destaques)]
fig5.add_trace(go.Scatter(
    x=df_dest["average_rating"],
    y=df_dest["log_ratings"],
    mode="markers+text",
    marker=dict(color="#FF5722", size=12),
    text=df_dest["title"].str[:15] + "...",
    textposition="top center",
    name="Destaques",
    textfont=dict(size=10),
))

fig5.update_layout(height=550, plot_bgcolor="white",
    xaxis=dict(showgrid=True, gridcolor="#f0f0f0"),
    yaxis=dict(showgrid=True, gridcolor="#f0f0f0"),
)
fig5.show()

In [18]:
from google.colab import files

fig1_novo.write_html("livros_g1_decadas.html")
fig2_novo.write_html("livros_g2_avaliacoes_decada.html")
fig3.write_html("livros_g3_coeficientes.html")
fig4.write_html("livros_g4_real_vs_previsto.html")
fig5.write_html("livros_g5_nota_vs_popularidade.html")

files.download("livros_g1_decadas.html")
files.download("livros_g2_avaliacoes_decada.html")
files.download("livros_g3_coeficientes.html")
files.download("livros_g4_real_vs_previsto.html")
files.download("livros_g5_nota_vs_popularidade.html")

print("✅ Todos os gráficos salvos!")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Todos os gráficos salvos!
